# S11 · BI/ML distribuido: KPIs + predicción del viento en vivo en Grafana
**Nayder Arce Mayta** · Pushgateway `localhost:49091` · Grafana `localhost:43000` → *Viento Juliaca — Pipeline U2*

1. KPIs batch del flujo (del Parquet de S09) y de la U1 (Gold histórico) → Pushgateway (PUT).
2. Predicción en vivo con el modelo de S10 → Pushgateway por `instance = sensorId`.
3. Auditoría: error real en producción (predicción vs lo que realmente pasó 1 h después).

In [1]:
import os, time, json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, IntegerType

BASE = os.getenv("VIENTO_BASE", "/opt")
ART = f"{BASE}/artifacts"
KAFKA = os.getenv("KAFKA_BOOTSTRAP", "kafka:9092")
TOPIC = "viento-eventos"
PUSHGATEWAY = os.getenv("PUSHGATEWAY", "http://pushgateway:9091")

spark = (SparkSession.builder
         .appName("s11-bi-ml-viento-nayder")
         .master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4040")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "|", time.strftime("%Y-%m-%d %H:%M:%S"))

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-04dc5890-a2c1-465a-8ef2-af67a9dad521;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Spark 4.2.0 | 2026-11-03 01:23:44


In [2]:
import requests

def push(job, instance, metricas, metodo="PUT"):
    """Publica un grupo de métricas en el Pushgateway (formato texto de Prometheus).
    PUT reemplaza el grupo completo (job, instance)."""
    cuerpo = "".join(f"{k} {float(v)}\n" for k, v in metricas.items() if v is not None)
    url = f"{PUSHGATEWAY}/metrics/job/{job}/instance/{instance}"
    r = requests.request(metodo, url, data=cuerpo, timeout=5)
    r.raise_for_status()
    return r.status_code

def borrar_grupo(job, instance):
    return requests.delete(f"{PUSHGATEWAY}/metrics/job/{job}/instance/{instance}", timeout=5).status_code

## 1. KPIs batch del flujo de eventos (volumen, validez, latencia)

In [3]:
obs = spark.read.parquet(f"{ART}/viento_eventos_observabilidad")
k = obs.agg(F.count("*").alias("total"), F.avg(F.col("isValid").cast("int")).alias("ratio"),
            F.avg("latencyMs").alias("lat"), F.countDistinct("sensorId", "obsTime").alias("lecturas_unicas")).first()
kpis_flujo = {"lambda26_bi_eventos_total": k["total"], "lambda26_bi_eventos_validos_ratio": k["ratio"],
              "lambda26_bi_latencia_promedio_ms": k["lat"], "lambda26_bi_lecturas_unicas": k["lecturas_unicas"]}
print(kpis_flujo, push("lambda26_bi_flujo", "viento_batch", kpis_flujo))

[Stage 1:===>                                                     (1 + 14) / 15]

{'lambda26_bi_eventos_total': 144, 'lambda26_bi_eventos_validos_ratio': 1.0, 'lambda26_bi_latencia_promedio_ms': 4982.041666666667, 'lambda26_bi_lecturas_unicas': 6} 200


## 2. KPIs de negocio de la capa batch (U1): climatología del mes actual

In [4]:
gold = spark.read.parquet(f"{ART}/viento_gold")
umbral = json.load(open(f"{ART}/umbral_viento.json"))["umbral_rafaga_kmh"]
mes = int(time.strftime("%m"))
g = gold.filter(F.col("mes") == mes).agg(F.avg("rafaga_max").alias("raf"), F.avg("dia_fuerte").alias("p"),
                                         F.avg("velocidad_media").alias("vel")).first()
kpis_u1 = {"lambda26_bi_u1_rafaga_max_media_mes": g["raf"], "lambda26_bi_u1_prob_dia_fuerte_mes": g["p"],
           "lambda26_bi_u1_velocidad_media_mes": g["vel"], "lambda26_bi_umbral_rafaga_kmh": umbral}
print(kpis_u1, push("lambda26_bi_batch", "u1_gold", kpis_u1))

{'lambda26_bi_u1_rafaga_max_media_mes': 38.57479166666667, 'lambda26_bi_u1_prob_dia_fuerte_mes': 0.23541666666666666, 'lambda26_bi_u1_velocidad_media_mes': 8.767979166666665, 'lambda26_bi_umbral_rafaga_kmh': 46} 200


## 3. Predicción en vivo → Pushgateway (`foreachBatch`)

In [10]:
# Contrato viento.lectura v1 (docs/contratos.md)
esquema_viento = (StructType()
    .add("tipoEvento", StringType()).add("version", IntegerType())
    .add("sensorId", StringType()).add("estacion", StringType())
    .add("lat", DoubleType()).add("lon", DoubleType())
    .add("velocidadViento", DoubleType()).add("direccionViento", DoubleType()).add("rafaga", DoubleType())
    .add("temperatura", DoubleType()).add("humedad", DoubleType()).add("presion", DoubleType())
    .add("obsTime", LongType()).add("origen", StringType()).add("timestamp", LongType()))

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA)
         .option("subscribe", TOPIC)
         .option("startingOffsets", "latest")
         .option("failOnDataLoss", "false")
         .load())

eventos = (crudo.select("partition", "offset", F.col("timestamp").alias("kafkaTs"),
                        F.col("value").cast("string").alias("json_str"))
           .select("partition", "offset", "kafkaTs", "json_str",
                   F.from_json("json_str", esquema_viento).alias("e"))
           .select("partition", "offset", "kafkaTs", "json_str", "e.*")
           .withColumn("ts", (F.col("timestamp") / 1000).cast("timestamp"))       # tiempo del evento (lectura)
           .withColumn("obs_ts", (F.col("obsTime") / 1000).cast("timestamp")))    # paso de 15 min del modelo
eventos.printSchema()

root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTs: timestamp (nullable = true)
 |-- json_str: string (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- version: integer (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- estacion: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocidadViento: double (nullable = true)
 |-- direccionViento: double (nullable = true)
 |-- rafaga: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- obsTime: long (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- obs_ts: timestamp (nullable = true)



In [6]:
from pyspark.ml import PipelineModel
modelo = PipelineModel.load(f"{ART}/modelo_viento_1h")      # de S10, sin reentrenar

def features(df):
    hora = F.hour("ts_feat") + F.minute("ts_feat") / 60.0
    return (df.withColumn("dir_sin", F.sin(F.radians("direccion"))).withColumn("dir_cos", F.cos(F.radians("direccion")))
              .withColumn("hora_sin", F.sin(2 * 3.141592653589793 * hora / 24))
              .withColumn("hora_cos", F.cos(2 * 3.141592653589793 * hora / 24))
              .withColumn("mes_sin", F.sin(2 * 3.141592653589793 * F.month("ts_feat") / 12))
              .withColumn("mes_cos", F.cos(2 * 3.141592653589793 * F.month("ts_feat") / 12)))

P_PRED = f"{ART}/predicciones_stream_s11"
entrada = (eventos.filter((F.col("origen") == "open-meteo") & F.col("velocidadViento").isNotNull())
           .select("sensorId", "estacion", "obs_ts", F.col("velocidadViento").alias("velocidad"), "rafaga",
                   F.col("direccionViento").alias("direccion"), "temperatura", "humedad", "presion"))

def en_vivo(df, batch_id):
    pred = (modelo.transform(features(df.dropDuplicates(["sensorId", "obs_ts"]).withColumn("ts_feat", F.col("obs_ts"))))
            .select(F.lit(batch_id).alias("batchId"), "sensorId", "estacion", F.col("obs_ts").alias("fechaHora"),
                    "velocidad", "rafaga", F.round("prediction", 2).alias("prediccion1h")))
    pred.write.mode("append").parquet(P_PRED)
    for r in pred.orderBy("fechaHora").collect():     # el último push por sensor = la lectura más reciente
        push("lambda26_ml_streaming", r["sensorId"], {
            "lambda26_ml_viento_real": r["velocidad"],
            "lambda26_ml_viento_prediccion_1h": r["prediccion1h"],
            "lambda26_ml_rafaga_real": r["rafaga"],
        })

consulta = (entrada.writeStream.foreachBatch(en_vivo)
            .option("checkpointLocation", f"{ART}/chk_s11_pred").trigger(processingTime="10 seconds").start())

## 4. Confirmar que es en vivo: dos consultas PromQL separadas 30 s
`push_time_seconds` cambia en cada micro-lote → prueba que el job sigue empujando (no es un valor congelado).
El viento real y la predicción cambian cuando Open-Meteo publica un paso nuevo (cada 15 min).

In [7]:
PROMETHEUS = os.getenv("PROMETHEUS", "http://prometheus:9090")
def promql(q):
    res = requests.get(f"{PROMETHEUS}/api/v1/query", params={"query": q}, timeout=5).json()["data"]["result"]
    return {r["metric"].get("instance", ""): round(float(r["value"][1]), 2) for r in res}

time.sleep(40)                                    # que haya al menos un push
for vuelta in (1, 2):
    print(f"--- consulta {vuelta} · {time.strftime('%H:%M:%S')}")
    print("  push_time_seconds         :", promql('push_time_seconds{job="lambda26_ml_streaming"}'))
    print("  lambda26_ml_viento_real   :", promql("lambda26_ml_viento_real"))
    print("  lambda26_ml_viento_pred_1h:", promql("lambda26_ml_viento_prediccion_1h"))
    if vuelta == 1: time.sleep(30)

--- consulta 1 · 01:24:41
  push_time_seconds         : {'juliaca-viento-01': 1793669072.75, 'lampa-viento-03': 1793669072.75, 'puno-viento-02': 1793669072.76}
  lambda26_ml_viento_real   : {'juliaca-viento-01': 5.2, 'lampa-viento-03': 7.6, 'puno-viento-02': 4.3}
  lambda26_ml_viento_pred_1h: {'juliaca-viento-01': 5.69, 'lampa-viento-03': 7.52, 'puno-viento-02': 7.23}
--- consulta 2 · 01:25:11
  push_time_seconds         : {'juliaca-viento-01': 1793669092.61, 'lampa-viento-03': 1793669092.61, 'puno-viento-02': 1793669092.62}
  lambda26_ml_viento_real   : {'juliaca-viento-01': 5.2, 'lampa-viento-03': 7.6, 'puno-viento-02': 4.3}
  lambda26_ml_viento_pred_1h: {'juliaca-viento-01': 5.69, 'lampa-viento-03': 7.52, 'puno-viento-02': 7.23}


## 5. Auditoría: error real en producción
Corre esta celda **a mano cuando el pipeline lleve ≥ 75 min funcionando** (se puede repetir; el stream sigue vivo).
Une cada predicción hecha a la hora T con la lectura real de T + 1 h. Las lecturas reales salen de todos los
Parquet que ya guarda el pipeline (observabilidad + predicciones).

In [11]:
def leer(ruta, cols):
    try: return spark.read.parquet(ruta).select(*cols)
    except Exception: return None

preds = [d for d in (leer(P_PRED, ["sensorId", "fechaHora", "velocidad", "prediccion1h"]),
                     leer(f"{ART}/predicciones_viento_s10", ["sensorId", "fechaHora", "velocidad",
                          F.col("prediccionVelocidad1h").alias("prediccion1h")])) if d is not None]
p = preds[0]
for d in preds[1:]: p = p.unionByName(d)
p = p.dropDuplicates(["sensorId", "fechaHora"])

reales = [d for d in (leer(f"{ART}/viento_eventos_observabilidad", [F.col("sensorId").alias("sid"), F.col("obs_ts").alias("fechaReal"),
                                                                   F.col("velocidadViento").alias("velocidadReal1h")]),
                      leer(P_PRED, [F.col("sensorId").alias("sid"), F.col("fechaHora").alias("fechaReal"),
                                    F.col("velocidad").alias("velocidadReal1h")])) if d is not None]
real = reales[0]
for d in reales[1:]: real = real.unionByName(d)
real = real.dropDuplicates(["sid", "fechaReal"])

audit = (p.alias("a").join(real.alias("b"), (F.col("a.sensorId") == F.col("b.sid"))
                           & (F.col("b.fechaReal") == F.col("a.fechaHora") + F.expr("INTERVAL 1 HOUR")))
         .select("a.sensorId", "a.fechaHora", "a.velocidad", "a.prediccion1h", "b.velocidadReal1h")
         .withColumn("errorAbsoluto", F.round(F.abs(F.col("prediccion1h") - F.col("velocidadReal1h")), 2))
         .withColumn("errorPersistencia", F.round(F.abs(F.col("velocidad") - F.col("velocidadReal1h")), 2)))
r = audit.agg(F.count("*").alias("n"), F.avg("errorAbsoluto").alias("mae"), F.avg("errorPersistencia").alias("mae_p")).first()
print(f"totalComparado={r['n']}  maeProduccion={r['mae']}  maePersistenciaProduccion={r['mae_p']}")
audit.orderBy(F.desc("fechaHora")).show(12)
if r["n"]:
    push("lambda26_ml_auditoria", "viento", {"lambda26_ml_mae_produccion": r["mae"],
                                              "lambda26_ml_mae_persistencia_produccion": r["mae_p"],
                                              "lambda26_ml_total_comparado": r["n"]})
else:
    print("Todavía no hay pares (predicción, real 1 h después). Vuelve a correr esta celda más tarde.")

26/11/03 04:20:15 ERROR Executor: Exception in task 0.0 in stage 7352.0 (TID 6754)
org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///opt/artifacts/predicciones_stream_s11/part-00000-25f6f9a1-6994-4797-a2f4-996226ac6a5d-c000.snappy.parquet.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:940)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:149)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:777)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apa

Py4JJavaError: An error occurred while calling o100073.collectToPython.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///opt/artifacts/predicciones_stream_s11/part-00000-25f6f9a1-6994-4797-a2f4-996226ac6a5d-c000.snappy.parquet.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:940)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:149)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:777)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark

## 6. Al terminar la demo: detener y limpiar el Pushgateway (quitar el # y correr)

In [ ]:
# consulta.stop()
# for s_ in ["juliaca-viento-01", "puno-viento-02", "lampa-viento-03"]:
#     print(s_, borrar_grupo("lambda26_ml_streaming", s_))

**Reflexión (Regla #37 de Google / por qué Pushgateway es un atajo):** el Pushgateway guarda el último
valor empujado y Prometheus lo sigue raspando aunque el job haya muerto → un productor caído se ve
como "valor estable". Para un job continuo lo correcto es exponer `/metrics` propio (pull), o al menos
publicar `push_time_seconds` y alertar si es viejo. Aquí lo usamos porque el notebook es un job efímero.

In [13]:
consulta.stop()

In [14]:
# 1) Compactar: miles de Parquet chiquitos → pocos archivos (solución al problema de archivos pequeños)
spark.conf.set("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)
for origen, destino, cols in [
    (f"{ART}/predicciones_stream_s11", f"{ART}/pred_s11_compacto", ["sensorId", "fechaHora", "velocidad", "prediccion1h"]),
    (f"{ART}/viento_eventos_observabilidad", f"{ART}/obs_compacto", ["sensorId", "obs_ts", "velocidadViento"]),
]:
    n_arch = len([f for f in os.listdir(origen) if f.endswith(".parquet")])
    spark.read.parquet(origen).select(*cols).dropDuplicates().coalesce(1).write.mode("overwrite").parquet(destino)
    print(f"{origen.split('/')[-1]}: {n_arch} archivos → 1")

# 2) Auditoría sobre los datos compactados
p = spark.read.parquet(f"{ART}/pred_s11_compacto").dropDuplicates(["sensorId", "fechaHora"])
real = (spark.read.parquet(f"{ART}/obs_compacto")
        .select(F.col("sensorId").alias("sid"), F.col("obs_ts").alias("fechaReal"), F.col("velocidadViento").alias("velocidadReal1h"))
        .dropDuplicates(["sid", "fechaReal"]))
audit = (p.alias("a").join(real.alias("b"), (F.col("a.sensorId") == F.col("b.sid"))
                           & (F.col("b.fechaReal") == F.col("a.fechaHora") + F.expr("INTERVAL 1 HOUR")))
         .select("a.sensorId", "a.fechaHora", "a.velocidad", "a.prediccion1h", "b.velocidadReal1h")
         .withColumn("errorAbsoluto", F.round(F.abs(F.col("prediccion1h") - F.col("velocidadReal1h")), 2))
         .withColumn("errorPersistencia", F.round(F.abs(F.col("velocidad") - F.col("velocidadReal1h")), 2)))
r = audit.agg(F.count("*").alias("n"), F.avg("errorAbsoluto").alias("mae"), F.avg("errorPersistencia").alias("mae_p")).first()
print(f"totalComparado={r['n']}  maeProduccion={r['mae']:.3f}  maePersistenciaProduccion={r['mae_p']:.3f}")
audit.groupBy("sensorId").agg(F.count("*").alias("n"), F.round(F.avg("errorAbsoluto"), 2).alias("mae_modelo"),
                              F.round(F.avg("errorPersistencia"), 2).alias("mae_persistencia")).show()
audit.orderBy(F.desc("fechaHora")).show(9)
push("lambda26_ml_auditoria", "viento", {"lambda26_ml_mae_produccion": r["mae"],
                                          "lambda26_ml_mae_persistencia_produccion": r["mae_p"],
                                          "lambda26_ml_total_comparado": r["n"]})

predicciones_stream_s11: 682 archivos → 1


viento_eventos_observabilidad: 2172 archivos → 1
totalComparado=27  maeProduccion=3.647  maePersistenciaProduccion=3.919
+-----------------+---+----------+----------------+
|         sensorId|  n|mae_modelo|mae_persistencia|
+-----------------+---+----------+----------------+
|juliaca-viento-01|  9|      3.52|            3.94|
|   puno-viento-02|  9|      3.23|            3.76|
|  lampa-viento-03|  9|      4.19|            4.06|
+-----------------+---+----------+----------------+

+-----------------+-------------------+---------+------------+---------------+-------------+-----------------+
|         sensorId|          fechaHora|velocidad|prediccion1h|velocidadReal1h|errorAbsoluto|errorPersistencia|
+-----------------+-------------------+---------+------------+---------------+-------------+-----------------+
|juliaca-viento-01|2026-11-02 22:15:00|      8.5|        9.42|            5.6|         3.82|              2.9|
|  lampa-viento-03|2026-11-02 22:15:00|      1.4|        4.19|        

200